In [2]:
from pyspark.sql.functions import (
    col, sha2, concat_ws, coalesce, lit, current_timestamp, current_date,
    to_date, year, month, quarter, dayofweek, dayofmonth,
    date_format, when, row_number, monotonically_increasing_id
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

StatementMeta(, 3d8f9a69-de21-4474-a61d-787b6c4874f2, 4, Finished, Available, Finished, False)

### Dim_Date

In [2]:
start_date = "2016-01-01"
end_date   = "2020-12-31"

df_calendar = spark.sql(f"""
    SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS full_date
""")

dim_date = (df_calendar
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("day", dayofmonth(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("year", year(col("full_date")))
    .withColumn("day_of_week", date_format(col("full_date"), "EEEE"))
    .withColumn("is_weekend", when(dayofweek(col("full_date")).isin(1,7), True).otherwise(False))
    .select("date_key", "full_date", "day", "month", "month_name",
            "quarter", "year", "day_of_week", "is_weekend")
)

dim_date.write.format("delta").mode("overwrite").saveAsTable("dwh.dim_date")
print(f" dim_date: {dim_date.count()} rows")

StatementMeta(, a00103c2-3ea8-4bac-85b2-ff37035a03db, 4, Finished, Available, Finished, False)

 dim_date: 1827 rows


### Dim_Payment_Type 

In [3]:
dim_payment_type = (spark.table("stg.stg_order_payments")
    .select("payment_type").distinct()
    .withColumn("payment_type_key", row_number().over(Window.orderBy("payment_type")))
    .select("payment_type_key", "payment_type")
)

dim_payment_type.write.format("delta").mode("overwrite").saveAsTable("dwh.dim_payment_type")
print(f" dim_payment_type: {dim_payment_type.count()} rows")

StatementMeta(, a00103c2-3ea8-4bac-85b2-ff37035a03db, 5, Finished, Available, Finished, False)

 dim_payment_type: 5 rows


### Dim_Order_Status 

In [4]:
dim_order_status = (spark.table("stg.stg_orders")
    .select("order_status").distinct()
    .withColumn("order_status_key", row_number().over(Window.orderBy("order_status")))
    .select("order_status_key", "order_status")
)

dim_order_status.write.format("delta").mode("overwrite").saveAsTable("dwh.dim_order_status")
print(f" dim_order_status: {dim_order_status.count()} rows")

StatementMeta(, a00103c2-3ea8-4bac-85b2-ff37035a03db, 6, Finished, Available, Finished, False)

 dim_order_status: 8 rows


### dim_customer (SCD2)

In [3]:
# قراءة الداتا من STG وحساب الـ row_hash
df_src = (spark.table("stg.stg_customers")
    .withColumn("row_hash", sha2(concat_ws("|",
        coalesce(col("customer_unique_id"), lit("")),
        coalesce(col("customer_zip_code_prefix"), lit("")),
        coalesce(col("customer_city"), lit("")),
        coalesce(col("customer_state"), lit(""))
    ), 256))
    .dropDuplicates(["customer_unique_id"])  # نتأكد إن كل شخص حقيقي له صف واحد
)

# لو أول مرة بس (الجدول لسه مش موجود) — Initial Load
if not spark.catalog.tableExists("dwh.dim_customer"):
    dim_customer_init = (df_src
        .withColumn("customer_key", row_number().over(Window.orderBy("customer_unique_id")))
        .withColumn("effective_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .select("customer_key", "customer_unique_id", "customer_zip_code_prefix",
                "customer_city", "customer_state", "row_hash",
                "effective_date", "end_date", "is_current")
    )
    dim_customer_init.write.format("delta").mode("overwrite").saveAsTable("dwh.dim_customer")
    print(f" Initial Load: {dim_customer_init.count()} customers")
else:
    print("dim_customer already exists — SCD2 Merge will run instead of the Initial Load")

StatementMeta(, 286de875-5857-4573-b359-6d61a201e7e6, 5, Finished, Available, Finished, False)

 Initial Load: 96096 customers


###  dim_customer ... (SCD2) Merge (To be executed in all future runs — Incremental Load)

In [5]:


dwh_customer = DeltaTable.forName(spark, "dwh.dim_customer")

# 1) أكبر customer_key حالي عشان نكمل الترقيم منه
max_key = spark.table("dwh.dim_customer").agg({"customer_key": "max"}).collect()[0][0] or 0

# 2) الريكوردز الحالية (is_current = true) بس من الـ target
df_current = spark.table("dwh.dim_customer").filter(col("is_current") == True)

# 3) تحديد الصفوف الجديدة كليًا أو اللي اتغيرت (Distributed بالكامل، من غير Action)
df_changes = (df_src.alias("src")
    .join(df_current.alias("tgt"), on="customer_unique_id", how="left")
    .filter(col("tgt.row_hash").isNull() | (col("tgt.row_hash") != col("src.row_hash")))
    .select("src.*")
)

window_spec = Window.orderBy("customer_unique_id")

# 4) صفوف الـ Insert الجديدة (customers جدد أو نسخ محدثة)
df_inserts = (df_changes
    .withColumn("customer_key", (row_number().over(window_spec) + lit(max_key)))
    .withColumn("effective_date", current_date())
    .withColumn("end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("merge_key", col("customer_unique_id"))   # مفتاح مطابقة عادي لعملية الـ Update
)

# 5) نفس الصفوف بـ merge_key = NULL عشان تجبر merge على INSERT (Staged Merge Pattern)
df_updates_trigger = df_inserts.withColumn("merge_key", lit(None).cast("string"))

df_staged = df_inserts.unionByName(df_updates_trigger)

# 6) عملية Merge واحدة بس — Atomic و Distributed بالكامل جوه الـ Cluster
(dwh_customer.alias("tgt")
    .merge(
        df_staged.alias("src"),
        "tgt.customer_unique_id = src.merge_key AND tgt.is_current = true"
    )
    .whenMatchedUpdate(set = {
        "is_current": lit(False),
        "end_date": current_date()
    })
    .whenNotMatchedInsert(values = {
        "customer_key": "src.customer_key",
        "customer_unique_id": "src.customer_unique_id",
        "customer_zip_code_prefix": "src.customer_zip_code_prefix",
        "customer_city": "src.customer_city",
        "customer_state": "src.customer_state",
        "row_hash": "src.row_hash",
        "effective_date": "src.effective_date",
        "end_date": "src.end_date",
        "is_current": "src.is_current"
    })
    .execute()
)

# 7) استخراج الـ Metrics من الـ Delta transaction log (مفيش Action زيادة على البيانات)
last_operation_metrics = (spark.sql("DESCRIBE HISTORY dwh.dim_customer LIMIT 1")
    .select("operationMetrics")
    .collect()[0][0]
)

print(f" SCD2 Merge executed on dim_customer")
print(f"    Expired rows (Updated - is_current=false): {last_operation_metrics.get('numTargetRowsUpdated', 0)}")
print(f"    New rows (Inserted): {last_operation_metrics.get('numTargetRowsInserted', 0)}")

StatementMeta(, 286de875-5857-4573-b359-6d61a201e7e6, 7, Finished, Available, Finished, False)

 SCD2 Merge executed on dim_customer
    Expired rows (Updated - is_current=false): 0
    New rows (Inserted): 0


### dim_seller (Initial Load + SCD2 Merge)

In [7]:
# ============================================================
# Cell 7 — dim_seller SCD2 (Initial Load + Incremental Merge)
# ============================================================

# حساب row_hash
df_src_seller = (spark.table("stg.stg_sellers")
    .withColumn("row_hash", sha2(concat_ws("|",
        coalesce(col("seller_id"), lit("")),
        coalesce(col("seller_zip_code_prefix"), lit("")),
        coalesce(col("seller_city"), lit("")),
        coalesce(col("seller_state"), lit(""))
    ), 256))
    .dropDuplicates(["seller_id"])
)

# Initial Load (أول مرة بس)
if not spark.catalog.tableExists("dwh.dim_seller"):
    dim_seller_init = (df_src_seller
        .withColumn("seller_key", row_number().over(Window.orderBy("seller_id")))
        .withColumn("effective_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .select("seller_key", "seller_id", "seller_zip_code_prefix",
                "seller_city", "seller_state", "row_hash",
                "effective_date", "end_date", "is_current")
    )
    dim_seller_init.write.format("delta").mode("overwrite").saveAsTable("dwh.dim_seller")
    print(f" Initial Load: {dim_seller_init.count()} sellers")

else:
    # SCD2 Incremental Merge
    dwh_seller = DeltaTable.forName(spark, "dwh.dim_seller")
    max_key = spark.table("dwh.dim_seller").agg({"seller_key": "max"}).collect()[0][0] or 0
    df_current_seller = spark.table("dwh.dim_seller").filter(col("is_current") == True)

    df_changes_seller = (df_src_seller.alias("src")
        .join(df_current_seller.alias("tgt"), on="seller_id", how="left")
        .filter(col("tgt.row_hash").isNull() | (col("tgt.row_hash") != col("src.row_hash")))
        .select("src.*")
    )

    window_spec = Window.orderBy("seller_id")

    df_inserts_seller = (df_changes_seller
        .withColumn("seller_key", (row_number().over(window_spec) + lit(max_key)))
        .withColumn("effective_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("merge_key", col("seller_id"))
    )

    df_updates_trigger_seller = df_inserts_seller.withColumn("merge_key", lit(None).cast("string"))
    df_staged_seller = df_inserts_seller.unionByName(df_updates_trigger_seller)

    (dwh_seller.alias("tgt")
        .merge(
            df_staged_seller.alias("src"),
            "tgt.seller_id = src.merge_key AND tgt.is_current = true"
        )
        .whenMatchedUpdate(set = {
            "is_current": lit(False),
            "end_date": current_date()
        })
        .whenNotMatchedInsert(values = {
            "seller_key": "src.seller_key",
            "seller_id": "src.seller_id",
            "seller_zip_code_prefix": "src.seller_zip_code_prefix",
            "seller_city": "src.seller_city",
            "seller_state": "src.seller_state",
            "row_hash": "src.row_hash",
            "effective_date": "src.effective_date",
            "end_date": "src.end_date",
            "is_current": "src.is_current"
        })
        .execute()
    )

    metrics = spark.sql("DESCRIBE HISTORY dwh.dim_seller LIMIT 1").select("operationMetrics").collect()[0][0]
    print(f" SCD2 Merge executed on dim_seller")
    print(f"    Updated: {metrics.get('numTargetRowsUpdated', 0)} |  Inserted: {metrics.get('numTargetRowsInserted', 0)}")

StatementMeta(, 286de875-5857-4573-b359-6d61a201e7e6, 9, Finished, Available, Finished, False)

 SCD2 Merge executed on dim_seller
    Updated: 0 |  Inserted: 0


### dim_product (Initial Load + SCD2 Merge)

In [9]:
# ============================================================
# Cell 8 — dim_product SCD2 (Initial Load + Incremental Merge)
# ============================================================

df_src_product = (spark.table("stg.stg_products")
    .withColumn("row_hash", sha2(concat_ws("|",
        coalesce(col("product_id"), lit("")),
        coalesce(col("product_category_name_english"), lit("")),
        coalesce(col("product_name_lenght").cast("string"), lit("")),
        coalesce(col("product_description_lenght").cast("string"), lit("")),
        coalesce(col("product_photos_qty").cast("string"), lit("")),
        coalesce(col("product_weight_g").cast("string"), lit("")),
        coalesce(col("product_length_cm").cast("string"), lit("")),
        coalesce(col("product_height_cm").cast("string"), lit("")),
        coalesce(col("product_width_cm").cast("string"), lit(""))
    ), 256))
    .dropDuplicates(["product_id"])
)

if not spark.catalog.tableExists("dwh.dim_product"):
    dim_product_init = (df_src_product
        .withColumn("product_key", row_number().over(Window.orderBy("product_id")))
        .withColumn("effective_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .select("product_key", "product_id", "product_category_name_english",
                "product_name_lenght", "product_description_lenght", "product_photos_qty",
                "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
                "row_hash", "effective_date", "end_date", "is_current")
    )
    dim_product_init.write.format("delta").mode("overwrite").saveAsTable("dwh.dim_product")
    print(f" Initial Load: {dim_product_init.count()} products")

else:
    dwh_product = DeltaTable.forName(spark, "dwh.dim_product")
    max_key = spark.table("dwh.dim_product").agg({"product_key": "max"}).collect()[0][0] or 0
    df_current_product = spark.table("dwh.dim_product").filter(col("is_current") == True)

    df_changes_product = (df_src_product.alias("src")
        .join(df_current_product.alias("tgt"), on="product_id", how="left")
        .filter(col("tgt.row_hash").isNull() | (col("tgt.row_hash") != col("src.row_hash")))
        .select("src.*")
    )

    window_spec = Window.orderBy("product_id")

    df_inserts_product = (df_changes_product
        .withColumn("product_key", (row_number().over(window_spec) + lit(max_key)))
        .withColumn("effective_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("merge_key", col("product_id"))
    )

    df_updates_trigger_product = df_inserts_product.withColumn("merge_key", lit(None).cast("string"))
    df_staged_product = df_inserts_product.unionByName(df_updates_trigger_product)

    (dwh_product.alias("tgt")
        .merge(
            df_staged_product.alias("src"),
            "tgt.product_id = src.merge_key AND tgt.is_current = true"
        )
        .whenMatchedUpdate(set = {
            "is_current": lit(False),
            "end_date": current_date()
        })
        .whenNotMatchedInsert(values = {
            "product_key": "src.product_key",
            "product_id": "src.product_id",
            "product_category_name_english": "src.product_category_name_english",
            "product_name_lenght": "src.product_name_lenght",
            "product_description_lenght": "src.product_description_lenght",
            "product_photos_qty": "src.product_photos_qty",
            "product_weight_g": "src.product_weight_g",
            "product_length_cm": "src.product_length_cm",
            "product_height_cm": "src.product_height_cm",
            "product_width_cm": "src.product_width_cm",
            "row_hash": "src.row_hash",
            "effective_date": "src.effective_date",
            "end_date": "src.end_date",
            "is_current": "src.is_current"
        })
        .execute()
    )

    metrics = spark.sql("DESCRIBE HISTORY dwh.dim_product LIMIT 1").select("operationMetrics").collect()[0][0]
    print(f" SCD2 Merge executed on dim_product")
    print(f"    Updated: {metrics.get('numTargetRowsUpdated', 0)} |  Inserted: {metrics.get('numTargetRowsInserted', 0)}")

StatementMeta(, 286de875-5857-4573-b359-6d61a201e7e6, 11, Finished, Available, Finished, False)

 SCD2 Merge executed on dim_product
    Updated: 0 |  Inserted: 0


### dim_review_text (Outrigger Dimension) , (SCD Type 1 + Incremental)

In [4]:
# ============================================================
# Cell — dim_review_text (Satellite table, Grain: order_id)
# ============================================================

df_src_review_text = (spark.table("stg.stg_order_reviews")
    .withColumn("rn", row_number().over(
        Window.partitionBy("order_id").orderBy(col("review_answer_timestamp").desc())
    ))
    .filter(col("rn") == 1)
    .select("order_id", "review_comment_title", "review_comment_message")
    .filter(col("review_comment_message").isNotNull() | col("review_comment_title").isNotNull())
    .withColumn("row_hash", sha2(concat_ws("|",
        coalesce(col("review_comment_title"), lit("")),
        coalesce(col("review_comment_message"), lit(""))
    ), 256))
)

if not spark.catalog.tableExists("dwh.dim_review_text"):
    df_src_review_text.write.format("delta").mode("overwrite").saveAsTable("dwh.dim_review_text")
    inserted_count = df_src_review_text.count()
    print(f" Initial Load: {df_src_review_text.count()} rows in dim_review_text")

else:
    dim_review_text_tbl = DeltaTable.forName(spark, "dwh.dim_review_text")
    (dim_review_text_tbl.alias("tgt")
        .merge(
            df_src_review_text.alias("src"),
            "tgt.order_id = src.order_id"
        )
        .whenMatchedUpdate(condition="tgt.row_hash != src.row_hash", set={
            "review_comment_title": "src.review_comment_title",
            "review_comment_message": "src.review_comment_message",
            "row_hash": "src.row_hash"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    metrics = spark.sql("DESCRIBE HISTORY dwh.dim_review_text LIMIT 1").select("operationMetrics").collect()[0][0]
    print(f"Merge executed on dim_review_text")
    print(f"    Updated: {metrics.get('numTargetRowsUpdated', 0)} | Inserted: {metrics.get('numTargetRowsInserted', 0)}")

StatementMeta(, 3d8f9a69-de21-4474-a61d-787b6c4874f2, 6, Finished, Available, Finished, False)

Merge executed on dim_review_text
    Updated: 0 | Inserted: 0


### fact_order_items (scd 1+ incremental load)

In [8]:
# ============================================================
# Cell 9 — fact_order_items (Transaction Fact, Grain: order_id + order_item_id)
# ============================================================

df_current_customers = spark.table("dwh.dim_customer").filter(col("is_current") == True) \
    .select("customer_key", col("customer_unique_id"))
df_current_sellers = spark.table("dwh.dim_seller").filter(col("is_current") == True) \
    .select("seller_key", "seller_id")
df_current_products = spark.table("dwh.dim_product").filter(col("is_current") == True) \
    .select("product_key", "product_id")

# ربط customer_id (بتاع الأوردر) بـ customer_unique_id عشان نلاقي الـ customer_key الصح
df_customer_lookup = (spark.table("stg.stg_customers")
    .select("customer_id", "customer_unique_id")
    .join(df_current_customers, on="customer_unique_id", how="inner")
    .select("customer_id", "customer_key")
)

df_src_items = (spark.table("stg.stg_order_items")
    .join(spark.table("stg.stg_orders").select("order_id", "customer_id"), on="order_id", how="left")
    .join(df_customer_lookup, on="customer_id", how="left")
    .join(df_current_sellers, on="seller_id", how="left")
    .join(df_current_products, on="product_id", how="left")
    .withColumn("purchase_date_key",
        date_format(col("shipping_limit_date"), "yyyyMMdd").cast("int"))  # بديل مؤقت، هنستبدلها بتاريخ الشراء الفعلي تحت
)

# نجيب تاريخ الشراء الحقيقي من الأوردر (مش من order_items)
df_src_items = (df_src_items
    .drop("purchase_date_key")
    .join(spark.table("stg.stg_orders").select("order_id", "order_purchase_timestamp"), on="order_id", how="left")
    .withColumn("purchase_date_key", date_format(col("order_purchase_timestamp"), "yyyyMMdd").cast("int"))
    .withColumn("shipping_limit_date_key", date_format(col("shipping_limit_date"), "yyyyMMdd").cast("int"))
    .withColumn("row_hash", sha2(concat_ws("|",
        coalesce(col("price").cast("string"), lit("")),
        coalesce(col("freight_value").cast("string"), lit(""))
    ), 256))
    .select("order_id", "order_item_id", "customer_key", "seller_key", "product_key",
            "purchase_date_key", "shipping_limit_date_key", "price", "freight_value", "row_hash")
)

if not spark.catalog.tableExists("dwh.fact_order_items"):
    df_src_items.write.format("delta").mode("overwrite").saveAsTable("dwh.fact_order_items")
    print(f" Initial Load: {df_src_items.count()} rows in fact_order_items")
else:
    fact_items_tbl = DeltaTable.forName(spark, "dwh.fact_order_items")
    (fact_items_tbl.alias("tgt")
        .merge(
            df_src_items.alias("src"),
            "tgt.order_id = src.order_id AND tgt.order_item_id = src.order_item_id"
        )
        .whenMatchedUpdate(condition="tgt.row_hash != src.row_hash", set={
            "price": "src.price",
            "freight_value": "src.freight_value",
            "row_hash": "src.row_hash"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    metrics = spark.sql("DESCRIBE HISTORY dwh.fact_order_items LIMIT 1").select("operationMetrics").collect()[0][0]
    print(f" Merge executed | Inserted: {metrics.get('numTargetRowsInserted',0)} | Updated: {metrics.get('numTargetRowsUpdated',0)}")

StatementMeta(, 3d8f9a69-de21-4474-a61d-787b6c4874f2, 10, Finished, Available, Finished, False)

 Merge executed | Inserted: 0 | Updated: 0


### fact_payments (scd 1+ incremental load)

In [13]:
# ============================================================
# Cell 10 — fact_payments (Transaction Fact, Grain: order_id + payment_sequential)
# ============================================================

df_payment_type = spark.table("dwh.dim_payment_type")

df_src_payments = (spark.table("stg.stg_order_payments").alias("p")
    .join(spark.table("stg.stg_orders").select("order_id", "customer_id", "order_purchase_timestamp"), on="order_id", how="left")
    .join(df_customer_lookup, on="customer_id", how="left")
    .join(df_payment_type, on="payment_type", how="left")
    .withColumn("purchase_date_key", date_format(col("order_purchase_timestamp"), "yyyyMMdd").cast("int"))
    .withColumn("row_hash", sha2(concat_ws("|",
        coalesce(col("payment_value").cast("string"), lit("")),
        coalesce(col("payment_installments").cast("string"), lit(""))
    ), 256))
    .select("order_id", "payment_sequential", "customer_key", "payment_type_key",
            "purchase_date_key", "payment_value", "payment_installments", "row_hash")
)

if not spark.catalog.tableExists("dwh.fact_payments"):
    df_src_payments.write.format("delta").mode("overwrite").saveAsTable("dwh.fact_payments")
    print(f" Initial Load: {df_src_payments.count()} rows in fact_payments")
else:
    fact_payments_tbl = DeltaTable.forName(spark, "dwh.fact_payments")
    (fact_payments_tbl.alias("tgt")
        .merge(
            df_src_payments.alias("src"),
            "tgt.order_id = src.order_id AND tgt.payment_sequential = src.payment_sequential"
        )
        .whenMatchedUpdate(condition="tgt.row_hash != src.row_hash", set={
            "payment_value": "src.payment_value",
            "payment_installments": "src.payment_installments",
            "row_hash": "src.row_hash"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    metrics = spark.sql("DESCRIBE HISTORY dwh.fact_payments LIMIT 1").select("operationMetrics").collect()[0][0]
    print(f" Merge executed | Inserted: {metrics.get('numTargetRowsInserted', 0)} | Updated: {metrics.get('numTargetRowsUpdated', 0)}")

StatementMeta(, 286de875-5857-4573-b359-6d61a201e7e6, 15, Finished, Available, Finished, False)

 Merge executed | Inserted: 0 | Updated: 0


### fact_orders (Accumulating Snapshot Fact)

In [10]:
# ============================================================
# Cell 11 — fact_orders (Accumulating Snapshot Fact, Grain: order_id)
# ============================================================
from pyspark.sql.functions import sum as spark_sum, count as spark_count, max as spark_max, datediff

df_order_status = spark.table("dwh.dim_order_status")

# Pre-aggregation قبل الـ join (عشان نتجنب Fan-out)
df_items_agg = (spark.table("stg.stg_order_items")
    .groupBy("order_id")
    .agg(
        spark_sum("price").alias("total_price"),
        spark_sum("freight_value").alias("total_freight_value"),
        spark_count("order_item_id").alias("total_items_count")
    )
)

df_payments_agg = (spark.table("stg.stg_order_payments")
    .groupBy("order_id")
    .agg(spark_sum("payment_value").alias("total_payment_value"))
)

# دلوقتي بس محتاجين review_score من الـ reviews (النصوص راحت لـ dim_review_text)
df_reviews_dedup = (spark.table("stg.stg_order_reviews")
    .withColumn("rn", row_number().over(Window.partitionBy("order_id").orderBy(col("review_answer_timestamp").desc())))
    .filter(col("rn") == 1)   # لو أكتر من review لنفس الأوردر، ناخد آخر واحد بس
    .select("order_id", "review_score", "review_creation_date", "review_answer_timestamp")
)

def date_key(c):
    return when(col(c).isNotNull(), date_format(col(c), "yyyyMMdd").cast("int")).otherwise(lit(None).cast("int"))

df_src_orders = (spark.table("stg.stg_orders").alias("o")
    .join(df_customer_lookup, on="customer_id", how="left")
    .join(df_order_status, on="order_status", how="left")
    .join(df_items_agg, on="order_id", how="left")
    .join(df_payments_agg, on="order_id", how="left")
    .join(df_reviews_dedup, on="order_id", how="left")
    .withColumn("purchase_date_key", date_key("order_purchase_timestamp"))
    .withColumn("approved_date_key", date_key("order_approved_at"))
    .withColumn("carrier_date_key", date_key("order_delivered_carrier_date"))
    .withColumn("delivered_date_key", date_key("order_delivered_customer_date"))
    .withColumn("estimated_delivery_date_key", date_key("order_estimated_delivery_date"))
    .withColumn("review_creation_date_key", date_key("review_creation_date"))
    .withColumn("review_answer_date_key", date_key("review_answer_timestamp"))
    .withColumn("approval_time_days", datediff(col("order_approved_at"), col("order_purchase_timestamp")))
    .withColumn("shipping_time_days", datediff(col("order_delivered_carrier_date"), col("order_approved_at")))
    .withColumn("delivery_time_days", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    .withColumn("delivery_delay_days", datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))
    .withColumn("row_hash", sha2(concat_ws("|",
        coalesce(col("order_status"), lit("")),
        coalesce(col("order_approved_at").cast("string"), lit("")),
        coalesce(col("order_delivered_carrier_date").cast("string"), lit("")),
        coalesce(col("order_delivered_customer_date").cast("string"), lit("")),
        coalesce(col("review_score").cast("string"), lit(""))
    ), 256))
    .select("order_id", "customer_key", "order_status_key",
            "purchase_date_key", "approved_date_key", "carrier_date_key",
            "delivered_date_key", "estimated_delivery_date_key",
            "review_creation_date_key", "review_answer_date_key",
            "total_price", "total_freight_value", "total_items_count", "total_payment_value",
            "approval_time_days", "shipping_time_days", "delivery_time_days", "delivery_delay_days",
            "review_score", "row_hash")
)

if not spark.catalog.tableExists("dwh.fact_orders"):
    df_src_orders.write.format("delta").mode("overwrite").saveAsTable("dwh.fact_orders")
    print(f" Initial Load: {df_src_orders.count()} rows in fact_orders")
else:
    fact_orders_tbl = DeltaTable.forName(spark, "dwh.fact_orders")
    (fact_orders_tbl.alias("tgt")
        .merge(
            df_src_orders.alias("src"),
            "tgt.order_id = src.order_id"
        )
        .whenMatchedUpdateAll(condition="tgt.row_hash != src.row_hash")
        .whenNotMatchedInsertAll()
        .execute()
    )
    metrics = spark.sql("DESCRIBE HISTORY dwh.fact_orders LIMIT 1").select("operationMetrics").collect()[0][0]
    print(f" Merge executed | Inserted: {metrics.get('numTargetRowsInserted',0)} | Updated: {metrics.get('numTargetRowsUpdated',0)}")

StatementMeta(, 3d8f9a69-de21-4474-a61d-787b6c4874f2, 12, Finished, Available, Finished, False)

 Merge executed | Inserted: 0 | Updated: 0
